# Import Commands
First import all necessary modules. Make sure that you have compiled the LCM basic algorithm available at https://research.nii.ac.jp/~uno/codes.htm and have added the executable to your PATH.

In [ ]:
import os
from adaptive_sampling.processing_tools.exploration import pattern_mining, extract_tools, refinement_tools
import shutil
from pathlib import Path
import glob
from ase.calculators.orca import OrcaProfile, ORCA
from ase.io import read, write

## Define an Orca Profile
This tells Orca where the binary is found in your system and is necessary due to the way Orca is interfaced to ASE. You may skip this step when using other electronic structure software.

In [ ]:
profile = OrcaProfile(command=shutil.which("orca"))

## Pattern Mining
First, we switch to the working directory containing the processed simulation data and organized in folders for trajectories, data frames, and reaction_lists.
Then we call the pattern mining routine (either for a single condition or for two conditions combined (see find_freq_patterns_pair function)) and set a collection name and a frequency threshold. 

In [ ]:
# first perform the pattern mining (here shown for a single simulation collection)

work_dir = "your_working_directory" # change to your working directory (use absolute path)
os.chdir(work_dir)

collection_name = "rct_collection_name"  # change to your collection name
pattern_mining.find_freq_patterns(f"{work_dir}/reactions_lists", f"{collection_name}", 2)

# Extract Trajectories Based on Reactive Patterns
Here, we first take a look at the obtained reactive patterns based on a given frequency.
Then we go through the simulation's trajectories and extract the trajectories of the reactive patterns.

In [ ]:
# then extract the raw trajectories for refinement

# the complete reactions are all patterns, 
# while rct_indices include only the pattern indices which are over the threshold
reactions, reactions_dict, complete_reactions, complete_reactions_dict, rct_indices = extract_tools.get_reaction_patterns(f"{work_dir}/{collection_name}.out.json", freq_threshold=7) # adjust freq threshold to a min freq of patterns appearing in simulations

extract_tools.extract_reactions(f"{work_dir}",
                 complete_reactions,
                 rct_indices) # we extract only those over the threshold

# Optimization of Reactants and Products
You have two options to optimize the start and end points for you trajectories. You either choose to use:
- the Sella optimizer followed by a vibrational frequency analysis
or
- a combined opt+freq calculation provided by your software of choice. 

See the example below for the first option. The combined calc example can be found at the end of this notebook.

In [ ]:
# start the refinement by optimizing all reactants and products
current_path = Path(os.getcwd())
for dir in (current_path.glob("pattern*")):
    print(dir)
    os.chdir(dir)

    charge = 0
    mult = 1
    # Read charge and multiplicity from the extracted pattern
    pattern = "charge_mult*"
    for filename in glob.glob(pattern):
        with open(filename, "r") as f:
            for line in f:
                charge, mult = map(int, line.strip().split("\t"))
    
    Path("start_opt").mkdir(exist_ok=True)
    Path("end_opt").mkdir(exist_ok=True)

    for file in dir.glob("pattern*.xyz"):
        extract_tools.extract_reactant_product(file, "start.xyz", "end.xyz")
        shutil.move("start.xyz", "start_opt/start.xyz")
        shutil.move("end.xyz", "end_opt/end.xyz")

    for mulliken_file in dir.glob("charge_mult*"):
        name = mulliken_file.name
        shutil.copy(mulliken_file, f"start_opt/{name}")
        shutil.copy(mulliken_file, f"end_opt/{name}")

    os.chdir("start_opt")
    calc = ORCA(profile=profile, charge=charge, mult=mult, orcasimpleinput='hf-3c engrad')
    refinement_tools.sella_opt(calculator=calc, mol_file="start.xyz", order=0)
    
    calc = ORCA(profile=profile, charge=charge, mult=mult, orcasimpleinput='hf-3c freq', orcablocks='%freq  Temp  39 end')
    refinement_tools.convert_geom_rm_X(input_file="opt_start.xyz.traj")
    refinement_tools.ase_calc(calculator=calc, mol_file="opt_start.xyz")


    os.chdir("../end_opt")
    calc = ORCA(profile=profile, charge=charge, mult=mult, orcasimpleinput='hf-3c engrad')
    refinement_tools.sella_opt(calculator=calc, mol_file="end.xyz", order=0)
    
    calc = ORCA(profile=profile, charge=charge, mult=mult, orcasimpleinput='hf-3c freq', orcablocks='%freq  Temp  39 end')
    refinement_tools.convert_geom_rm_X(input_file="opt_end.xyz.traj")
    refinement_tools.ase_calc(calculator=calc, mol_file="opt_end.xyz")

os.chdir("../..")

# Perform Minima Frequency Check 
Check if a minimum point has been found and move the pattern directories to MIN_FOUND accordingly.

In [ ]:
refinement_tools.find_minima()

# Prepare Input Files for DE-GSM Using the PyGSM Module
This input files are tailored for Orca in combination with PyGSM. If you use a different electronic structure software you may need to adjust the lot_inp_file and the package keywords according to the PyGSM documentation.

In [ ]:
# here change to MIN_FOUND directory and generate ostart and gsm_input files to copy them to the pattern directories
os.chdir("MIN_FOUND")
refinement_tools.write_file("! hf-3c engrad", f"{os.getcwd()}/ostart")
refinement_tools.write_file("-xyzfile reordered_init_geom.xyz -num_nodes 6 -reactant_geom_fixed -product_geom_fixed -package Orca -lot_inp_file ostart -ID 42", f"{os.getcwd()}/gsm_input")

# Start the Double-Ended Growing String Minimum Energy Path Searches
After the stationary points have been confirmed, you can start the double-ended GSM searches using the cell below. Please adjust the setting in the cell above to match your wishes.

In [ ]:
current_path = Path(os.getcwd())
for i, dir in enumerate(current_path.glob("pattern*")):
    print(dir)
    os.chdir(dir)
    shutil.copy('../ostart', 'ostart')
    shutil.copy('../gsm_input', 'gsm_input')

    with open('init_geom.xyz', 'w') as outfile:
        for filename in ['start_opt/opt_start.xyz', 'end_opt/opt_end.xyz']:
            with open(filename, 'r') as infile:
                outfile.write(infile.read())

    refinement_tools.rearrange_top("init_geom.xyz")

    print(i)
    refinement_tools.de_gsm(add_args_file='gsm_input', out_file='log', index=i)

os.chdir("..")

# Identify Reactions for Final Transition State Optimization
Identify viable reactions based on barrier thresholds.

In [ ]:
refinement_tools.find_viable_rcts(min_barrier=0.001, max_barrier=50.0)

# Optimization of the Transition States
After the viable reactions have been identified based on your settings, the transition states can be optimized.
This can be again done, as for the stationary points, either with the Sella optimizer, or by using the combined opt+freq of your electronic structure software of choice.
Please refer to the example cell at the end of this notebook for the second option.

In [ ]:
# optimize the TS nodes

current_path = Path(os.getcwd())
for dir in (current_path.glob("pattern*")):
    print(dir)
    os.chdir(dir)
    charge = 0
    mult = 1
    # Read charge and multiplicity from the extracted pattern
    pattern = "charge_mult*"
    for filename in glob.glob(pattern):
        with open(filename, "r") as f:
            for line in f:
                charge, mult = map(int, line.strip().split("\t"))
    
    calc = ORCA(profile=profile, charge=charge, mult=mult, orcasimpleinput='hf-3c engrad')
    os.chdir("ts_opt")
    current_dir = Path(os.getcwd())

    refinement_tools.sella_opt(calculator=calc, mol_file="ts.xyz", order=1)

    calc = ORCA(profile=profile, charge=charge, mult=mult, orcasimpleinput='hf-3c freq', orcablocks='%freq  Temp  39 end')
    refinement_tools.convert_geom_rm_X(input_file=f"opt_ts.xyz.traj")
    refinement_tools.ase_calc(calculator=calc, mol_file=f"opt_ts.xyz")

os.chdir("../..")

# Find Successful Reactions and Generate the Refined Reaction List (Network)
A reaction is deemed successful if all stationary points have been optimized and confirmed by frequency analysis and the reaction barrier found in the DE-GSM calculation does not exceed the imposed threshold.

In [ ]:
refinement_tools.find_successful_reactions(search_term="imaginary perturbations")

# Additional Examples with Orca

In [ ]:
# opt and freq directly with orca
print(os.getcwd()) # you have to be in the 'extracted' directory
current_path = Path(os.getcwd())
for dir in (current_path.glob("pattern*")):
    print(dir)
    os.chdir(dir)

    charge = 0
    mult = 1
    # Read charge and multiplicity from the extracted pattern
    pattern = "charge_mult*"
    for filename in glob.glob(pattern):
        with open(filename, "r") as f:
            for line in f:
                charge, mult = map(int, line.strip().split("\t"))

    calc = ORCA(profile=profile, charge=charge, mult=mult, orcasimpleinput='hf-3c opt freq', orcablocks='%freq  Temp  39 end')

    Path("start_opt").mkdir(exist_ok=True)
    Path("end_opt").mkdir(exist_ok=True)

    for file in dir.glob("pattern*.xyz"):
        extract_tools.extract_reactant_product(file, "start.xyz", "end.xyz")
        shutil.move("start.xyz", "start_opt/start.xyz")
        shutil.move("end.xyz", "end_opt/end.xyz")

    for mulliken_file in dir.glob("charge_mult*"):
        name = mulliken_file.name
        shutil.copy(mulliken_file, f"start_opt/{name}")
        shutil.copy(mulliken_file, f"end_opt/{name}")

    os.chdir("start_opt")
    refinement_tools.ase_calc(calculator=calc, mol_file="start.xyz")

    os.chdir("../end_opt")
    refinement_tools.ase_calc(calculator=calc, mol_file="end.xyz")

os.chdir("../..")
refinement_tools.find_minima()

In [ ]:
# opt and freq directly with orca for transition states
print(os.getcwd()) # you have to be in the 'VIABLE_RCTS' directory

current_path = Path(os.getcwd())
for dir in (current_path.glob("pattern*")):
    print(dir)
    os.chdir(dir)

    os.chdir("ts_opt")
    calc = ORCA(profile=profile, orcasimpleinput='hf-3c optts freq', orcablocks='%freq  Temp  39 end')
    for file in dir.glob("TSnode*.xyz"):
        refinement_tools.ase_calc(calculator=calc, mol_file=file)

os.chdir("../..")